In [1]:
import pandas as pd
import json

# 1. Load data
df = pd.read_csv('project-1-at-2026-04-15-16-19-821d8960.csv')

# Daftar aspek sesuai rencana penelitian
ASPECTS = ['PIGMENTATION', 'LONGEVITY', 'TEXTURE', 'HYDRATION', 'PRICE']

def parse_labels(label_str):
    if pd.isna(label_str) or label_str == '[]':
        return {}
    try:
        data = json.loads(label_str)
        results = {}
        for item in data:
            for lbl in item['labels']:
                if '-' in lbl:
                    aspect, sentiment = lbl.split('-', 1)
                    if aspect in ASPECTS and aspect not in results:
                        results[aspect] = sentiment
        return results
    except:
        return {}

# 2. Flattening data
flattened = []
for _, row in df.iterrows():
    labels_dict = parse_labels(row['label'])
    for aspect in ASPECTS:
        flattened.append({
            'id': row['id'],
            'review_text': row['review_text'],
            'annotator': row['annotator'],
            'aspect': aspect,
            'sentiment': labels_dict.get(aspect, "None")
        })

flat_df = pd.DataFrame(flattened)

# 3. Fungsi Majority Voting (Minimal 2 dari 3)
def get_majority(series):
    # Menghitung frekuensi setiap label
    counts = series.value_counts()
    majority_val = counts.idxmax()
    
    # Cek berapa orang yang memberikan label untuk baris ini
    jumlah_anotator = len(series) 
    
    if jumlah_anotator >= 3:
        # KASUS SHARED SUBSET: Harus ada minimal 2 orang setuju
        if counts.max() >= 2:
            return majority_val
        else:
            return "Ambiguous" # Terjadi jika 1 vs 1 vs 1
    else:
        # KASUS INDIVIDUAL PARTITION: Ambil label yang ada
        # (Karena jumlah_anotator = 1)
        return majority_val

# 4. Eksekusi Voting dan Pivot
voting_df = flat_df.groupby(['id', 'review_text', 'aspect'])['sentiment'].apply(get_majority).reset_index()
final_df = voting_df.pivot(index=['id', 'review_text'], columns='aspect', values='sentiment').reset_index()

# Simpan hasil
final_df.to_csv('final_majority_voting_results.csv', index=False)